# Hyper-parameter Tuning

### Learning Objectives

In this notebook we learn to:

1. Given Linear Regression or Decision Tree model, distinguish between it's *parameters* and *hyper-parameters*
2. Recognize the 5 elements of hyper-parameter search in scikit-learn
3. Explain the process of hyper-parameter search; describing the role of each of the 5 elements
4. Explain the difference between Grid Search and either Random Search or Bayesian Search
5. Recognize the more efficient, model-specific cross-validations


## Hyper-parameters vs. Parameters

**Parameters** are what models learn during `.fit(X, y)`.

**Hyper-parameters** are passed to the constructor function of the estimator (model).

To find the names and current values for all parameters for a given estimator, `estimator.get_params()` like so:

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': 0.0,
 'max_iter': 100,
 'n_jobs': None,
 'penalty': 'deprecated',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

### Transformer Hyper-parameters

In `sklearn`, transformers are estimators too, and have their own HPs.

For example preprocessing steps like `PolynomialFeatures` is an estimator with `degree: int` (among others):

In [10]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures()
poly.get_params()

{'degree': 2, 'include_bias': True, 'interaction_only': False, 'order': 'C'}

Like wise, an imputer is an estimator with different `strategy` (`"mean"`, `"median"`, ..etc.):

In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(
    strategy='mean' # or 'median', 'most_frequent', 'constant'
)
imputer.get_params()

{'add_indicator': False,
 'copy': True,
 'fill_value': None,
 'keep_empty_features': False,
 'missing_values': nan,
 'strategy': 'mean'}

## Hyper-parameter tuning methods

It is possible and **recommended to search the hyper-parameter space for the best** [**cross validation**](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation) **score**.

The five elements of HP search are:

1. an **estimator**
2. a **parameter space**
3. a **search method** (tuning algorithms)
4. a **cross-validation scheme**
5. a [**scoring function**](https://scikit-learn.org/stable/modules/grid_search.html#gridsearch-scoring) (examplee: `accuracy_score` in classification)

## Search method and parameter space

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV "sklearn.model_selection.GridSearchCV") exhaustively considers all parameter combinations

[`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV "sklearn.model_selection.RandomizedSearchCV") can sample a given number of candidates from a parameter space with a specified distribution.

While Grid and Random search are "blind" strategies, [`skopt.BayesSearchCV`](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html) is "informed". It builds a probability model (a **surrogate model**) that updates it's sampling distribution based on previous trials. On some datasets, it leads to the same score with: **7x fewer iterations and 5x less time**.

![](../assets/hp_search_methods.png)

See also: [Population based training | Google Deepmind](https://deepmind.google/blog/population-based-training-of-neural-networks/).

### Efficient Model-specific CV

Some models can fit data for a range of values of some parameter almost as efficiently as fitting the estimator for a single value of the parameter. This feature can be leveraged to perform a more efficient cross-validation used for model selection of this parameter. Examples are:

- `RidgeCV` for regression
- `LogisticRegressionCV` for classification

See: [Model specific cross-validation](https://scikit-learn.org/stable/modules/grid_search.html#model-specific-cross-validation)

## HP Tuning Workflow

```{mermaid}
graph TD
    %% Define color styling for nodes
    classDef purpleNode fill:#ccccff,stroke:#000,stroke-width:1px,color:#000;

    %% Define the nodes
    Parameters(param_grid):::purpleNode
    Dataset(data):::purpleNode
    CV(Search):::purpleNode
    Trainingdata(train_data):::purpleNode
    Testdata(test_data):::purpleNode
    Retrainedmodel(search.best_estimator_):::purpleNode
    Finalevaluation(".score()"):::purpleNode

    %% Define the connections
    Dataset --> Trainingdata
    Dataset --> Testdata
    Parameters --> CV
    Trainingdata --> CV
    Trainingdata --> Retrainedmodel
    CV --> Retrainedmodel
    Retrainedmodel --> Finalevaluation
    Testdata --> Finalevaluation
```

First and foremost in this workflow is:

- The `data` is split intro `train_data` and `test_data`
- The search space `param_grid` is specified

Then:

1. Hyper-parameter search algorithm `GridSearchCV` iterate on the train-validation splits; for each model doing training and scoring.
2. The resulting `search.best_params` are used to initialize a new model and fit it again on the entire train set.
3. Finally, the trained model is evaluated on the test set.

### Example: Randomized Search (cross-validated)

In the following example, we randomly search over the parameter space of a random forest with a [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV) object.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from scipy.stats import randint

In [ ]:
# create a synthetic dataset
X, y = make_regression(
    n_samples=20640,
    n_features=8,
    noise=0.1,
    random_state=0
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

In [ ]:
# define the parameter space that will be searched over
param_distributions = {
    'n_estimators': randint(1, 5),
    'max_depth': randint(5, 10)
}
# now create a searchCV object and fit it to the data
search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=0),
    n_iter=5,
    param_distributions=param_distributions,
    random_state=0
)

search.fit(X_train, y_train)

In [ ]:
model = search.best_estimator_
model

array([ 284.79557583,   52.55187449,   36.64321571, ..., -136.91520557,
        126.28826424,  116.76904048], shape=(5160,))

In [ ]:
model.score(X_test, y_test)

![Figure: Cross Validation](../assets/inner_cross_validation.png)

## Pipelines are also Estimators

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV "sklearn.model_selection.GridSearchCV") and [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV "sklearn.model_selection.RandomizedSearchCV") allow searching over parameters of composite or nested estimators such as [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline "sklearn.pipeline.Pipeline"), [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html#sklearn.compose.ColumnTransformer "sklearn.compose.ColumnTransformer") using a dedicated `<estimator>__<parameter>` syntax:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import make_moons

In [ ]:
# 1. Prepare Data
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

In [ ]:
# 2. Define the Pipeline
# We treat the scaler and the classifier as a single "mega-model."
from sklearn.impute import SimpleImputer


pipe = Pipeline([
    ('imputer', SimpleImputer()),
    ('model', RandomForestClassifier())
])

In [ ]:

# 3. Define the Search Grid
# Syntax: <step_name>__<parameter_name>
param_grid = {
    'imputer__strategy': ['mean', 'median'],
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5, None]
}

search = GridSearchCV(pipe, param_grid, cv=5)

In [ ]:
# 4. Run the Grid Search
# This automatically scales the data and trains the model for every combination.
search.fit(X_train, y_train)

print(f"Best settings found: {search.best_params_}")
print(f"Accuracy of best model: {search.best_score_:.2%}")

Best settings found: {'model__max_depth': 5, 'model__n_estimators': 100}
Accuracy of best model: 89.60%


In [ ]:
best_model = search.best_estimator_
best_model

In [ ]:
best_model.score(X_test, y_test)

Refer to [Pipeline: chaining estimators](https://scikit-learn.org/stable/modules/compose.html#pipeline) for more details.